# 무역 트렌드와 산업생산의 대조 — 정본 계산기

`연구계획서.md`의 둘째 갈래. 성질별 수출 실적으로 직접 만든 품목군 월 계열에
국가데이터처 광공업생산지수(`fact_ip`)를 붙여 수출과 생산의 성장률·동조성·선후행·브리지 회귀를 잰다. 수출 척도는 명목과 코드 단위 Törnqvist 물량이고
(단가 = 명목 − 물량), 한국은행 수출물가지수(2026-09-03부터 `cache/fact_xmpi.parquet`, DB에서는 뺌)로 실질화한 값과 수출출하지수는 참고 열로만 둔다(논문에 없음).
대응표는 계획서 §VII.1 표 A(2026-09-02 확정, 논문 표 2)를 코드로 옮긴 것이다.

| 절 | 내용 |
|---|---|
| §0 | 설정. 첫 갈래 캐시, DB(읽기전용), 국면 |
| §1 | 대응표(논문 표 2)와 생산지수 구축, 코드 단위 Törnqvist 물량. 수출물가·수출출하지수는 참고 |
| §2 | 성장률. 31년·국면별 명목·단가·물량과 생산지수 (표 3~5) |
| §2b | 달력 설명변수. 조업일수와 설·추석 세 창의 Δ12 (논문 III.2) |
| §3 | 동조성과 선후행. 달력 효과를 뺀 물량·명목 대 생산지수 Δ12 상관, 국면별, 교차상관 (표 4, 그림 2) |
| §4b | 공표 시차와 브리지 회귀. t+1월 1일·중순 정보 집합의 표본외 성적 (표 5). |
| §5 | 그림 1~3 |
| §6 | 검증. 논문 『관세청 수출 통계는 같은 달의 산업생산을 얼마나 먼저 알려 주는가』의 인용 수치 |

읽는 것: DB(읽기전용), 품목군 대응표 `outputs/품목군_대응표_수출.csv`, 수출물가 사본 `cache/fact_xmpi.parquet`·`dim_xmpi.parquet`.
다른 노트북의 산출물은 읽지 않는다(2026-09-14 독립화). §2b 끝의 대조 셀만 옛 캐시가 있으면 월 금액·중량을 맞대 본다.

실행: `jupyter nbconvert --to notebook --execute --inplace trend-vs-industrial-production.ipynb --ExecutePreprocessor.kernel_name=kcsdb`

## §0. 설정

In [1]:
import os, warnings
import numpy as np, pandas as pd, duckdb
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")):
    up = os.path.dirname(ROOT)
    if up == ROOT: raise FileNotFoundError("KCSDB2 루트를 찾지 못했습니다")
    ROOT = up
DB   = os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")
TT = os.path.join(ROOT, "research", "trade_trend")   # 무역 트렌드 연구 폴더(하위 폴더 셋)
HERE = os.path.join(TT, "industrial-production")
CACHE = os.path.join(HERE, "cache"); OUT = os.path.join(HERE, "outputs"); IMG = os.path.join(HERE, "img")
MAP = pd.read_csv(os.path.join(OUT, "품목군_대응표_수출.csv"), dtype=str)
con = duckdb.connect(DB, read_only=True)

PHASES = [("P1", 1995, 2000), ("P2", 2001, 2007), ("P3", 2008, 2015), ("P4", 2016, 2019), ("P5", 2020, 2023), ("P6", 2024, 2026)]
PH_LABEL = {p: f"{a}~{b if b < 2026 else '2026.07'}" for p, a, b in PHASES}
def yrs(a, b): return (b - a) if b < 2026 else (2025 + 7/12 - a)  # 2026 끝점은 2025.08~2026.07 합. 2023년 창과의 간격은 2.58년
def phase_of(ym):
    y = ym // 100
    for p, a, b in PHASES:
        if a <= y <= b: return p
EXP15 = [s for s in MAP.group15.unique() if s != "나머지"]
# 월 금액·중량: 성질별 수출 실적(153개 코드)을 품목군 대응표로 묶는다. 다른 노트북의 캐시에 기대지 않는다(2026-09-14 독립화).
_ex = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수출'").df().merge(MAP[["temper_cd", "group15"]], on="temper_cd", how="left")
assert _ex.group15.notna().all(), "대응표에 없는 수출 코드가 있다"
_gm = _ex.groupby(["yyyymm", "group15"])[["dlr", "wgt"]].sum().unstack()
W, Q = _gm["dlr"].copy(), _gm["wgt"].copy()
for M_ in (W, Q):
    M_.index = M_.index.astype(int); M_.columns.name = None
    M_["총수출"] = M_[EXP15 + ["나머지"]].sum(axis=1)
YM0, YM1 = int(W.index.min()), int(W.index.max())
ORDER = ["총수출", "비반도체"] + sorted(EXP15, key=lambda s: -W.loc[202301:202512, s].sum())
print("계열", len(W.columns), "| 기간", YM0, "~", YM1, "| 품목군", len(EXP15))

계열 17 | 기간 199501 ~ 202607 | 품목군 15


## §1. 대응표와 지수 구축

표 A를 코드로 둔다. 생산지수는 소분류 둘을 묶을 때 국가데이터처의 올해 가중치로, 수출물가지수는 한국은행의 2020년 가중치로
가중평균한다. 상위 항목에서 하위 하나를 빼는 자리(비반도체, 화공품에서 의약품)는 $(w_A I_A - w_B I_B)/(w_A - w_B)$로 한다.
연쇄 라스파이레스 지수를 고정 가중치로 다시 묶는 것이라 근사다. 의약품 수출물가지수는 2015.12부터라 그 전의 화공품은 화학제품 전체를 쓴다.

In [2]:
# group15 → (생산지수 더할 코드, 뺄 코드, 수출물가 더할 코드, 뺄 코드, 수출출하지수 코드(있을 때만))
MAP2 = {
    "총수출":     (["0"],            [],       ["*AA"],                 [],          ["0"]),
    "비반도체": ([],               [],       [],                      [],          ["0"]),   # 아래에서 따로 만든다
    "반도체":     (["C261"],         [],       ["30911AA"],             [],          ["C261"]),
    "승용차":     (["C301"],         [],       ["312111AA"],            [],          []),
    "자동차부품":  (["C303"],         [],       ["31213AA"],             [],          []),
    "석유제품":    (["C192"],         [],       ["3041AA"],              [],          ["C19"]),
    "철강제품":    (["C24"],          [],       ["307AA"],               [],          ["C24"]),
    "선박":       (["C311"],         [],       [],                      [],          ["C31"]),
    "무선통신기기": (["C264"],         [],       ["30951AA"],             [],          ["C264"]),
    "컴퓨터주변기기": (["C263"],       [],       ["3094AA"],              [],          ["C263"]),
    "정밀기기":    (["C27"],          [],       ["3096AA"],              [],          ["C27"]),
    "가전제품":    (["C285", "C265"], [],       ["31015AA", "30952AA"],  [],          []),
    "일반기계":    (["C29"],          [],       ["311AA"],               [],          ["C29"]),
    "화공품":     (["C20", "C22"],   [],       ["305AA"],               ["3054AA"],  ["C20", "C22"]),
    "의약품":     (["C212"],         [],       ["305411AA"],            [],          ["C21"]),
    "제조장비":    (["C292"],         [],       ["31124AA"],             [],          []),
    "이차전지":    (["C282"],         [],       ["310131AA"],            [],          []),
}
assert set(EXP15) <= set(MAP2), set(EXP15) - set(MAP2)

# 생산지수: 지표별 (yyyymm × ksic)
IPraw = con.execute("SELECT yyyymm, ksic, measure, value FROM fact_ip").df()
IP = {m: d.pivot(index="yyyymm", columns="ksic", values="value") for m, d in IPraw.groupby("measure")}
KW = con.execute("SELECT ksic, wgt FROM dim_ksic").df().set_index("ksic").wgt.to_dict()
KN = con.execute("SELECT ksic, name_ko FROM dim_ksic").df().set_index("ksic").name_ko.to_dict()
# 수출물가지수 달러기준: (yyyymm × item_cd)
# 수출물가지수는 2026-09-03에 DB에서 뺐다(참고 계산에만 쓰므로). 뺄 때 떠 둔 cache/*.parquet에서 읽는다.
XP = duckdb.sql(f"SELECT yyyymm, item_cd, value FROM '{CACHE.replace(chr(92), '/')}/fact_xmpi.parquet' WHERE imexp='수출' AND basis='D'").df() \
        .pivot(index="yyyymm", columns="item_cd", values="value")
XW = duckdb.sql(f"SELECT item_cd, wgt FROM '{CACHE.replace(chr(92), '/')}/dim_xmpi.parquet' WHERE imexp='수출'").df().set_index("item_cd").wgt.to_dict()
XN = duckdb.sql(f"SELECT item_cd, name_ko FROM '{CACHE.replace(chr(92), '/')}/dim_xmpi.parquet' WHERE imexp='수출'").df().set_index("item_cd").name_ko.to_dict()

def agg_idx(P, wd, plus, minus):
    """고정 가중 합성. 뺄 항목이 결측인 달은 더할 항목만으로 둔다(의약품 2015.12 이전)."""
    if not plus: return pd.Series(np.nan, index=P.index)
    num = sum(wd[c] * P[c] for c in plus); den = sum(wd[c] for c in plus)
    if minus:
        sub = sum(wd[c] * P[c] for c in minus); dsub = sum(wd[c] for c in minus)
        out = (num - sub) / (den - dsub)
        return out.where(sub.notna(), num / den)
    return num / den

IPX = pd.DataFrame({g: agg_idx(IP["prod"],    KW, a, b) for g, (a, b, _, _, _) in MAP2.items()}).loc[YM0:YM1]
PX  = pd.DataFrame({g: agg_idx(XP, XW, c, d)           for g, (_, _, c, d, _) in MAP2.items()}).loc[YM0:YM1]
XS  = pd.DataFrame({g: agg_idx(IP["ship_exp"], KW, e, []) for g, (_, _, _, _, e) in MAP2.items()}).loc[YM0:YM1]
for df in (IPX, PX, XS): df.index = df.index.astype(int)
W["비반도체"] = W["총수출"] - W["반도체"]; Q["비반도체"] = Q["총수출"] - Q["반도체"]

# 비반도체 생산지수: 총지수에서 C261을 빼는 대신, C261을 뺀 나머지 코드(중분류 전부 + C26의 다른 소분류)를 올해
# 가중치로 묶는다. 2026년 가중치로 빼는 식 (w0·I0 − w261·I261)/(w0 − w261)은 연쇄지수에는 근사이고,
# 반도체처럼 상대 수준이 크게 움직인 항목에서는 초기 연도가 왜곡된다. 두 방식을 견줘 둔다.
DK = con.execute("SELECT ksic, level, parent FROM dim_ksic").df()
EX_ALL = [c for c in DK[DK.level == 2].ksic if c != "C26"] + ["C262", "C263", "C264", "C265"]
# 2020년 신설 중분류(C34)는 1995~2019년이 없다. 전 기간 있는 코드만 묶는다.
def full_codes(M): return [c for c in EX_ALL if c in M.columns and M[c].loc[YM0:YM1].notna().all()]
EX_SEMI_CODES = full_codes(IP["prod"])
_drop = [c for c in EX_ALL if c not in EX_SEMI_CODES]
print(f"비반도체 합성: {len(EX_SEMI_CODES)}개 코드, 뺀 코드 {_drop} 가중치 합 {sum(KW[c] for c in _drop):.1f}/10000")
IPX["비반도체"] = agg_idx(IP["prod"],    KW, EX_SEMI_CODES, []).loc[YM0:YM1].values
ipx_sub = agg_idx(IP["prod"], KW, ["0"], ["C261"]).loc[YM0:YM1]
_c = np.corrcoef(np.log(IPX["비반도체"]).diff(12).dropna(), np.log(ipx_sub).diff(12).dropna())[0, 1]
print(f"비반도체 생산지수: 합성 대 뺄셈의 Δ12 상관 {_c:.3f}, 2020 평균 {IPX['비반도체'].loc[202001:202012].mean():.1f}")

# 비반도체 수출물가지수: 가격이 있는 열세 품목군(선박 제외)의 연쇄 Törnqvist. 가중치는 연도별 수출액 비중.
# 2020년 고정 가중치로 총지수에서 반도체를 빼면 1990년대에 음수가 나온다 — 반도체 가격이 100배 넘게 떨어졌기 때문.
PRICED = [g for g in EXP15 if g not in ("반도체", "선박")]
def chain_tornqvist(groups):
    sh = W[groups].groupby(W.index // 100).sum(); sh = sh.div(sh.sum(axis=1), axis=0)
    dlp = np.log(PX[groups]).diff()
    yr = pd.Series(dlp.index // 100, index=dlp.index)
    w_cur = sh.reindex(yr.values).set_index(dlp.index); w_prev = sh.reindex(yr.values - 1).set_index(dlp.index)
    w_prev = w_prev.fillna(w_cur)
    wbar = 0.5 * (w_cur + w_prev)
    ok = dlp.notna()
    step = (wbar.where(ok) * dlp).sum(axis=1) / wbar.where(ok).sum(axis=1)   # 결측 품목(의약품·제조장비 초기)은 그 달 가중치에서 뺀다
    lp = step.fillna(0).cumsum()
    idx = np.exp(lp); return idx / idx.loc[202001:202012].mean() * 100
PX["비반도체"] = chain_tornqvist(PRICED)
px_all = chain_tornqvist(PRICED + ["반도체"])
_c = np.corrcoef(np.log(px_all).diff(12).dropna(), np.log(PX["총수출"]).diff(12).dropna())[0, 1]
print(f"열네 품목군 연쇄 물가지수 대 한국은행 총지수: Δ12 상관 {_c:.3f}, 31년 연평균 {np.log(px_all.iloc[-1]/px_all.iloc[0])*100/yrs(1995,2026):.2f}% 대 "
      f"{np.log(PX['총수출'].iloc[-1]/PX['총수출'].iloc[0])*100/yrs(1995,2026):.2f}%")
XS["비반도체"] = agg_idx(IP["ship_exp"], KW, ["0"], ["C261"]).loc[YM0:YM1].values   # 수출출하는 소분류가 C26뿐이라 뺄셈으로
R = (W[list(MAP2)] / PX * 100)                     # 실질 수출 (2020년 달러)

# 단가와 물량은 품목군 중량을 그대로 쓰지 않고 첫 갈래와 같은 코드 단위 Törnqvist로 가른다.
# 품목군 중량은 값이 작고 무거운 코드에 끌려간다. 반도체 중량은 2024년에 한 해 만에 3분의 1이 됐는데,
# 금액 비중 1%인 기타 개별소자(44307)의 중량이 186천 톤에서 41천 톤으로 준 것이고 메모리는 5.5천 톤 그대로다.
raw = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수출'").df().merge(MAP[["temper_cd", "group15"]], on="temper_cd")
GROUP_CODES = {g: MAP.loc[MAP.group15 == g, "temper_cd"].tolist() for g in EXP15}
GROUP_CODES["총수출"] = MAP.temper_cd.tolist(); GROUP_CODES["비반도체"] = MAP.loc[MAP.group15 != "반도체", "temper_cd"].tolist()
def tornqvist(d0, d1):
    """d0·d1: temper_cd 인덱스, 열 dlr·wgt. 반환 (Δln V, Δln P, Δln Q), 로그 차이 그대로."""
    both = d0.join(d1, lsuffix="0", rsuffix="1", how="inner")
    both = both[(both.dlr0 > 0) & (both.dlr1 > 0) & (both.wgt0 > 0) & (both.wgt1 > 0)]
    w = 0.5 * (both.dlr0 / both.dlr0.sum() + both.dlr1 / both.dlr1.sum())
    dlp = (w * np.log((both.dlr1 / both.wgt1) / (both.dlr0 / both.wgt0))).sum()
    dlv = np.log(d1.dlr.sum() / d0.dlr.sum())
    return dlv, dlp, dlv - dlp
# 월 연쇄 물량지수 (2020=100): 달마다 앞 달과의 Törnqvist 단가 변화를 빼서 잇는다
MV = raw.pivot(index="yyyymm", columns="temper_cd", values="dlr").fillna(0); MW = raw.pivot(index="yyyymm", columns="temper_cd", values="wgt").fillna(0)
def chain_q(codes):
    v, w = MV[codes], MW[codes]
    ok = (v > 0) & (w > 0) & (v.shift(1) > 0) & (w.shift(1) > 0)
    sh = v.div(v.sum(axis=1), axis=0); wbar = 0.5 * (sh + sh.shift(1))
    dlp_i = np.log((v / w) / (v.shift(1) / w.shift(1)))
    dlp = (wbar.where(ok) * dlp_i.where(ok)).sum(axis=1) / wbar.where(ok).sum(axis=1)
    dlv = np.log(v.sum(axis=1) / v.sum(axis=1).shift(1))
    lq = (dlv - dlp).fillna(0).cumsum(); q = np.exp(lq)
    return q / q.loc[202001:202012].mean() * 100
QT = pd.DataFrame({g: chain_q(c) for g, c in GROUP_CODES.items()}).loc[YM0:YM1]
print("월 연쇄 Törnqvist 물량지수 31년 연평균(%):", {g: round(np.log(QT[g].loc[202508:202607].mean() / QT[g].loc[199501:199512].mean()) * 100 / yrs(1995, 2026), 1) for g in ["총수출", "비반도체", "반도체"]})

rows = []
for g, (a, b, c, d, e) in MAP2.items():
    if g == "비반도체":
        rows.append({"품목군": g, "생산지수": "C26을 뺀 모든 중분류와 C262~C265의 가중 합성", "수출물가지수": "열세 품목군의 연쇄 Törnqvist(수출액 가중)",
                     "수출출하지수": "0 −C261", "실질 첫 달": int(R[g].first_valid_index())}); continue
    rows.append({"품목군": g, "생산지수": "+".join(f"{k} {KN[k]}" for k in a) + ("".join(f" −{k} {KN[k]}" for k in b)),
                 "수출물가지수": "+".join(f"{k} {XN[k]}" for k in c) + ("".join(f" −{k} {XN[k]}" for k in d)) if c else "없음",
                 "수출출하지수": "+".join(e) if e else "없음",
                 "실질 첫 달": int(R[g].first_valid_index()) if R[g].notna().any() else -1})
TAB_MAP = pd.DataFrame(rows).set_index("품목군")
TAB_MAP.to_csv(os.path.join(OUT, "품목군_대응표_산업물가.csv"), encoding="utf-8-sig")
print(TAB_MAP.to_string())

비반도체 합성: 31개 코드, 뺀 코드 ['C34'] 가중치 합 85.8/10000


비반도체 생산지수: 합성 대 뺄셈의 Δ12 상관 0.929, 2020 평균 100.0
열네 품목군 연쇄 물가지수 대 한국은행 총지수: Δ12 상관 0.950, 31년 연평균 -2.70% 대 -0.76%
월 연쇄 Törnqvist 물량지수 31년 연평균(%): {'총수출': np.float64(3.0), '비반도체': np.float64(3.1), '반도체': np.float64(1.0)}
                                                     생산지수                           수출물가지수   수출출하지수  실질 첫 달
품목군                                                                                                        
총수출                                                 0 총지수                          *AA 총지수        0  199501
비반도체                      C26을 뺀 모든 중분류와 C262~C265의 가중 합성     열세 품목군의 연쇄 Törnqvist(수출액 가중)  0 −C261  199501
반도체                                          C261 반도체 제조업                      30911AA 반도체     C261  199501
승용차                                C301 자동차용 엔진 및 자동차 제조업                     312111AA 승용차       없음  199501
자동차부품                                  C303 자동차 신품 부품 제조업                    31213AA 자동차부품       없음  199501
석유제품                     

## §2. 명목·단가·물량과 생산 성장률의 비교

국면 끝점(직전 국면 마지막 해 → 이 국면 마지막 해)의 연간값으로 연평균 로그 변화를 낸다. 금액·중량은 연간 합, 지수는 연간 평균이다.
2026은 2025.08~2026.07의 12개월이다. 명목 $\Delta \ln V$, 코드 단위 Törnqvist 단가 $\Delta \ln U$, 물량 $\Delta \ln Q = \Delta \ln V - \Delta \ln U$(논문 III.1)와
생산지수의 연평균 성장률을 나란히 둔다(논문 표 3). 수출물가 $\Delta \ln P$, 실질 $\Delta \ln V - \Delta \ln P$, 단가−물가, 수출출하지수는 참고 열이다.

In [3]:
def annual(M, how):
    """월 표를 연 표로. 2026은 2025.08~2026.07."""
    y = M.loc[:202512].groupby(M.loc[:202512].index // 100).agg(how)
    ttm = M.loc[202508:202607].agg(how).rename(2026)
    return pd.concat([y, ttm.to_frame().T])
Vy = annual(W[list(MAP2)], "sum")
Py, IPy, XSy = annual(PX, "mean"), annual(IPX, "mean"), annual(XS, "mean")
Ry = Vy / Py * 100
# 코드 단위 연간 금액·중량 (끝점용). 2026은 2025.08~2026.07의 12개월 합.
ann = raw[raw.yyyymm <= 202512].assign(year=lambda d: d.yyyymm // 100).groupby(["year", "temper_cd"])[["dlr", "wgt"]].sum().reset_index()
ttm = raw[(raw.yyyymm >= 202508) & (raw.yyyymm <= 202607)].groupby("temper_cd")[["dlr", "wgt"]].sum().reset_index().assign(year=2026)
A = pd.concat([ann, ttm], ignore_index=True).set_index(["year", "temper_cd"])
def tq(g, y0, y1):
    d0 = A.loc[y0].reindex(GROUP_CODES[g]).dropna(); d1 = A.loc[y1].reindex(GROUP_CODES[g]).dropna()
    return tornqvist(d0[["dlr", "wgt"]], d1[["dlr", "wgt"]])

rows = []
for p, a, b in PHASES:
    y0 = a - 1 if a > 1995 else 1995; y1 = b; n = yrs(y0, y1)
    for g in MAP2:
        dl = lambda T: (np.log(T.loc[y1, g] / T.loc[y0, g]) * 100 / n) if (g in T and pd.notna(T.loc[y0, g]) and pd.notna(T.loc[y1, g])) else np.nan
        dV, dP, dIP, dXS = dl(Vy), dl(Py), dl(IPy), dl(XSy)
        _, dU, dQ = [x * 100 / n for x in tq(g, y0, y1)]
        rows.append({"국면": PH_LABEL[p], "계열": g, "명목": dV, "물가": dP, "실질": dV - dP, "물량": dQ,
                     "단가": dU, "단가−물가": dU - dP, "생산지수": dIP, "수출출하지수": dXS})
TAB_REAL = pd.DataFrame(rows)
TAB_REAL.to_csv(os.path.join(OUT, "tab2_real_growth.csv"), index=False, encoding="utf-8-sig")
for col in ["실질", "물량", "생산지수", "단가−물가"]:
    print(f"\n[{col} 연평균 %]")
    print(TAB_REAL.pivot(index="계열", columns="국면", values=col).reindex(ORDER)[list(PH_LABEL.values())].round(1).to_string())


[실질 연평균 %]
국면       1995~2000  2001~2007  2008~2015  2016~2019  2020~2023  2024~2026.07
계열                                                                          
총수출           12.5       11.7        6.9        1.5        3.0           8.6
비반도체          10.7       10.9        6.7       -1.1        2.2           3.2
반도체           47.0       31.0       19.0       16.6       14.4          11.3
화공품           15.0        6.8        5.7        3.6        2.6          -2.9
승용차           11.7       12.4        2.4       -0.5       12.6           2.3
일반기계           7.3       14.8        5.3        2.4        1.9          -4.0
석유제품          18.0        0.6        6.5        2.3       -2.8          -0.4
철강제품           8.8        2.7        6.1       -1.3       -2.4          -4.6
선박             NaN        NaN        NaN        NaN        NaN           NaN
자동차부품         20.2       22.8        9.3       -4.2       -0.8          -7.6
무선통신기기        44.9       34.8        1.8       13.7        0.4  

In [4]:
# 31년 전체(1995→2026.07): 실질 수출 대 생산지수 — 어느 품목군이 생산보다 빨리 자랐나
n_all = yrs(1995, 2026)
full = pd.DataFrame({
    "명목": np.log(Vy.loc[2026] / Vy.loc[1995]) * 100 / n_all,
    "물가": np.log(Py.loc[2026] / Py.loc[1995]) * 100 / n_all,
    "단가": pd.Series({g: tq(g, 1995, 2026)[1] * 100 / n_all for g in MAP2}),
    "물량": pd.Series({g: tq(g, 1995, 2026)[2] * 100 / n_all for g in MAP2}),
    "생산지수": np.log(IPy.loc[2026] / IPy.loc[1995]) * 100 / n_all,
    "수출출하지수": np.log(XSy.loc[2026] / XSy.loc[1995]) * 100 / n_all})
full["실질"] = full["명목"] - full["물가"]
full["단가−물가"] = full["단가"] - full["물가"]; full["실질−생산"] = full["실질"] - full["생산지수"]; full["물량−생산"] = full["물량"] - full["생산지수"]
TAB_FULL = full.reindex(ORDER)[["명목", "물가", "실질", "단가", "물량", "단가−물가", "생산지수", "실질−생산", "물량−생산", "수출출하지수"]]
TAB_FULL.to_csv(os.path.join(OUT, "tab2_full_growth.csv"), encoding="utf-8-sig")
print(TAB_FULL.round(1).to_string())

           명목    물가    실질    단가    물량  단가−물가  생산지수  실질−생산  물량−생산  수출출하지수
총수출       6.5  -1.4   7.9   4.5   2.0    5.9   4.4    3.4   -2.5     7.3
비반도체      5.6  -0.8   6.4   3.0   2.5    3.9   1.9    4.5    0.6     6.2
반도체       9.5 -15.3  24.8   9.2   0.3   24.5  19.8    5.0  -19.5    20.3
화공품       6.9   0.9   6.1   1.4   5.5    0.6   1.9    4.1    3.6     3.4
승용차       7.7   0.5   7.1   1.7   5.9    1.2   2.3    4.9    3.7     NaN
일반기계      6.3   0.1   6.2   1.5   4.8    1.4   2.8    3.4    2.0     6.0
석유제품     10.2   5.5   4.7   5.1   5.1   -0.5   2.1    2.6    3.1     4.9
철강제품      5.1   2.3   2.8   1.7   3.5   -0.7   1.5    1.3    2.0     3.9
선박        5.9   NaN   NaN   7.3  -1.4    NaN   4.8    NaN   -6.2     5.1
자동차부품    10.0   0.3   9.6   1.0   9.0    0.6   5.1    4.5    3.9     NaN
무선통신기기    9.7  -8.4  18.1   8.5   1.2   16.9   3.8   14.2   -2.7     6.4
컴퓨터주변기기   6.7  -7.8  14.5  10.9  -4.2   18.6   2.9   11.6   -7.0     4.4
정밀기기      8.7  -2.2  10.9   2.5   6.2    4.7   2.9 

## §2b. 달력 설명변수 (논문 III.2)

원계열의 전년 동월 대비에도 두 해의 조업일수 차이와 설·추석의 달 이동은 남는다. 그 달의 달력은 미리 알 수 있으므로
조업일수(월~금, 공휴일 제외)의 로그와 설·추석 세 창(연휴 앞 20일, 연휴 사흘, 연휴 뒤 10일에 든 날의 몫)을 설명변수로 만든다.
달력월 평균으로 중심화하고 Δ12를 취한다. 동조성은 수출과 생산 각각에서 이 일곱 변수로 설명되는 부분을 회귀로 뺀 잔차끼리 재고,
브리지 회귀는 모든 모형에 이 일곱 변수를 넣는다. 끝 셀은 옛 캐시(기술분석 노트북 산출물)가 있으면 월 금액·중량을 맞대 보기만 한다.

In [5]:
import datetime as dt
from korean_lunar_calendar import KoreanLunarCalendar
LEAD = 12
YM_LEAD = (YM1 // 100 + (YM1 % 100 + LEAD - 1) // 12) * 100 + (YM1 % 100 + LEAD - 1) % 12 + 1
CAL_VARS = ["ln_wd", "seol_pre", "seol_dur", "seol_post", "chu_pre", "chu_dur", "chu_post"]
wd_ = con.execute("SELECT base_ym AS ym, SUM(workdays) AS workdays FROM dim_workday10d GROUP BY 1 ORDER BY 1").df().set_index("ym")
def _lunar(y, m, d):
    k = KoreanLunarCalendar(); k.setLunarDate(y, m, d, False); return dt.date.fromisoformat(k.SolarIsoFormat())
def _hol_reg(anchor, w_pre, w_post, name):
    rows = []
    for y, d0 in anchor.items():
        dur = [d0 + dt.timedelta(k) for k in (-1, 0, 1)]
        pre = [dur[0] - dt.timedelta(k) for k in range(1, w_pre + 1)]; post = [dur[-1] + dt.timedelta(k) for k in range(1, w_post + 1)]
        for kk, win in [("pre", pre), ("dur", dur), ("post", post)]:
            for d in win: rows.append({"ym": d.year * 100 + d.month, "var": f"{name}_{kk}", "w": 1 / len(win)})
    return pd.DataFrame(rows).groupby(["ym", "var"])["w"].sum().unstack(fill_value=0.0)
_H = _hol_reg({y: _lunar(y, 1, 1) for y in range(1994, 2028)}, 20, 10, "seol").add(_hol_reg({y: _lunar(y, 8, 15) for y in range(1994, 2028)}, 20, 10, "chu"), fill_value=0.0)
def cal_regressors(start):
    """달력 설명변수를 start~YM_LEAD로 자르고 start~YM1의 달력월 평균으로 중심화한다."""
    X_ = wd_.join(_H).fillna(0.0); X_["ln_wd"] = np.log(X_["workdays"]); X_ = X_.loc[start:YM_LEAD].copy(); X_["month"] = X_.index % 100
    for c in CAL_VARS:
        mm = X_.loc[:YM1].groupby("month")[c].mean(); X_[c] = X_[c] - X_["month"].map(mm)
    return X_[CAL_VARS]
Xcal_exp = cal_regressors(YM0)
dCAL = Xcal_exp.diff(12)
print("달력 설명변수:", CAL_VARS, "| Δ12 범위", dCAL.dropna().index.min(), "~", dCAL.dropna().index.max())

달력 설명변수: ['ln_wd', 'seol_pre', 'seol_dur', 'seol_post', 'chu_pre', 'chu_dur', 'chu_post'] | Δ12 범위 199601 ~ 202707


In [6]:
# 옛 캐시와의 대조(출력만): 독립화 전에는 기술분석 노트북의 캐시를 읽었다. 파일이 있으면 맞대 보고, 차이가 있으면 원인을 볼 것.
def _cmp(label, new, path, col):
    if not os.path.exists(path): print(f"{label}: 옛 캐시 없음, 건너뜀"); return
    old = pd.read_csv(path).replace({"series": {"반도체 제외": "비반도체"}}).pivot(index="yyyymm", columns="series", values=col); old.index = old.index.astype(int)   # 옛 이름으로 만든 파일도 읽는다
    cols = [c for c in new.columns if c in old.columns]
    print(f"{label}: 맞댄 계열 {len(cols)} | 최대 절대 차이 {(new[cols] - old[cols].reindex(new.index)).abs().max().max():.3e}")
_cmp("월 금액", W[["총수출"] + EXP15], os.path.join(TT, "descriptive", "cache", "series_monthly.csv"), "dlr")
_cmp("월 중량", Q[["총수출"] + EXP15], os.path.join(TT, "descriptive", "cache", "series_monthly.csv"), "wgt")

월 금액: 맞댄 계열 16 | 최대 절대 차이 0.000e+00
월 중량: 맞댄 계열 16 | 최대 절대 차이 0.000e+00


## §3. 동조성과 선후행

전년 동월 대비 로그 차이 $\Delta_{12}$에서 달력 설명변수로 설명되는 부분을 전체 기간 회귀로 뺀 잔차의 상관. 물량(코드 단위 Törnqvist 월 연쇄)과
명목 수출을 생산지수(원지수)와 맞대고, 국면별은 물량 대 생산지수만 낸다(논문 표 4). 선후행은 같은 잔차의 교차상관이 가장 높은 시차로 본다.
실질 수출(수출물가로 나눈 것)과 수출출하지수의 상관은 참고 열이다.

In [7]:
def d12(M): return np.log(M).diff(12)
dR, dQ_, dV, dIP, dXS = d12(R), d12(QT.reindex(columns=list(MAP2))), d12(W[list(MAP2)]), d12(IPX), d12(XS)
def decal(M):
    """각 열의 Δ12에서 달력 설명변수로 설명되는 부분을 전체 기간 OLS로 뺀 잔차."""
    out = {}
    for c in M.columns:
        d = pd.DataFrame({"y": M[c]}).join(dCAL).dropna()
        if len(d) < 36: out[c] = M[c] * np.nan; continue
        out[c] = sm.OLS(d.y, sm.add_constant(d[CAL_VARS])).fit().resid.reindex(M.index)
    return pd.DataFrame(out)
eR, eQ, eV, eIP, eXS = decal(dR), decal(dQ_), decal(dV), decal(dIP), decal(dXS)
CAL_R2 = pd.DataFrame({lab: 1 - E.var() / D.where(E.notna()).var() for lab, E, D in [("물량", eQ, dQ_), ("명목", eV, dV), ("생산", eIP, dIP)]})
print("달력이 설명하는 Δ12 분산 몫:"); print(CAL_R2.loc[["총수출", "비반도체", "반도체"]].round(3).to_string())
ph = pd.Series([phase_of(x) for x in R.index], index=R.index)
rows = []
for g in MAP2:
    r = {"계열": g}
    def cc(x, y):
        j = pd.concat([x, y], axis=1).dropna()
        return (j.iloc[:, 0].corr(j.iloc[:, 1]), len(j)) if len(j) > 24 else (np.nan, len(j))
    r["물량·생산"], r["n"] = cc(eQ[g], eIP[g])
    r["명목·생산"], _ = cc(eV[g], eIP[g])
    r["실질·생산(참고)"], _ = cc(eR[g], eIP[g])
    r["물량·수출출하(참고)"], _ = cc(eQ[g], eXS[g])
    for p, a, b in PHASES:
        m = ph == p
        r[PH_LABEL[p]], _ = cc(eQ[g][m], eIP[g][m])       # 국면별은 물량 대 생산
        r[PH_LABEL[p] + "(실질,참고)"], _ = cc(eR[g][m], eIP[g][m])
    rows.append(r)
TAB_SYNC2 = pd.DataFrame(rows).set_index("계열").reindex(ORDER)
TAB_SYNC2.to_csv(os.path.join(OUT, "tab2_sync.csv"), encoding="utf-8-sig")
print(TAB_SYNC2.round(2).to_string())

달력이 설명하는 Δ12 분산 몫:
         물량     명목     생산
총수출   0.193  0.036  0.126
비반도체  0.225  0.061  0.219
반도체   0.016  0.010  0.005
         물량·생산    n  명목·생산  실질·생산(참고)  물량·수출출하(참고)  1995~2000  1995~2000(실질,참고)  2001~2007  2001~2007(실질,참고)  2008~2015  2008~2015(실질,참고)  2016~2019  2016~2019(실질,참고)  2020~2023  2020~2023(실질,참고)  2024~2026.07  2024~2026.07(실질,참고)
계열                                                                                                                                                                                                                                    
총수출       0.54  367   0.63       0.61         0.69      -0.01              0.35       0.72              0.66       0.80              0.78       0.05              0.38       0.71              0.75          0.36                 0.09
비반도체      0.46  367   0.64       0.42         0.68      -0.04              0.03       0.60              0.56       0.83              0.63       0.46              0.50       0.81       

In [8]:
# 선후행: 달력을 뺀 Δ12 잔차의 교차상관 corr(x_{t-k}, ip_t), k=-12..12. k>0이면 수출이 앞선다.
def xcorr(x, y, K=12):
    out = {}
    for k in range(-K, K + 1):
        j = pd.concat([x.shift(k), y], axis=1).dropna()
        out[k] = j.iloc[:, 0].corr(j.iloc[:, 1]) if len(j) > 24 else np.nan
    return pd.Series(out)
rows = []
for g in MAP2:
    xq, xv = xcorr(eQ[g], eIP[g]), xcorr(eV[g], eIP[g])
    rows.append({"계열": g, "물량 최대 시차": int(xq.idxmax()), "물량 최대 상관": xq.max(), "물량 동시 상관": xq[0],
                 "명목 최대 시차": int(xv.idxmax()), "명목 최대 상관": xv.max(), "명목 동시 상관": xv[0]})
TAB_XC = pd.DataFrame(rows).set_index("계열").reindex(ORDER)
TAB_XC.to_csv(os.path.join(OUT, "tab2_xcorr_d12.csv"), encoding="utf-8-sig")
print(TAB_XC.round(2).to_string())
XC = pd.DataFrame({g: xcorr(eQ[g], eIP[g]) for g in ["총수출", "비반도체", "반도체"]})
print(XC.round(2).T.to_string())

         물량 최대 시차  물량 최대 상관  물량 동시 상관  명목 최대 시차  명목 최대 상관  명목 동시 상관
계열                                                                 
총수출             0      0.54      0.54        -1      0.63      0.63
비반도체            0      0.46      0.46         0      0.64      0.64
반도체             0      0.35      0.35         0      0.51      0.51
화공품            -9      0.22      0.22        -2      0.59      0.56
승용차             0      0.66      0.66         0      0.72      0.72
일반기계            0      0.45      0.45        -3      0.57      0.55
석유제품            0      0.56      0.56         0      0.40      0.40
철강제품           -6      0.32      0.20        -5      0.48      0.41
선박              0      0.18      0.18         0      0.18      0.18
자동차부품           0      0.54      0.54         0      0.56      0.56
무선통신기기          0      0.45      0.45         0      0.48      0.48
컴퓨터주변기기         0      0.45      0.45        -3      0.28      0.27
정밀기기            0      0.11      0.11         0 

## §4b. 공표 시차와 브리지 회귀

수출과 생산이 같은 달에 움직이더라도 공표 시점이 다르면 먼저 나오는 쪽이 나중 것을 미리 짐작하게 한다. $t$월 자료의 공표는
월간 수출입 동향 $t+1$월 1일, 관세청 확정치(중량 포함) $t+1$월 중순, 광공업생산지수(산업활동동향) $t+1$월 마지막 영업일이다.
그래서 $t+1$월 1일과 중순에는 $t$월 수출과 $t-1$월 생산이 있다.
브리지 회귀 $\Delta_{12} Y_t = a + b\,\Delta_{12} Y_{t-h} + \gamma' \Delta_{12} \mathrm{cal}_t + c\,\Delta_{12} x_t$로 그 정보 집합에서 $t$월 생산을 예측하고, 수출 항이 없는
기준 모형과 확장창 표본외 RMSE를 견준다. 표본외는 2010.01부터다. 기준은 전월 생산만 쓴 것과 여기에 전월 경기선행지수를 더한 것 둘이다(논문 표 6).
관세청 10일 잠정치($t$월 11일·21일)는 논문 범위 밖이라 다루지 않는다.
**정보 집합 규칙**: 각 시점에 공표된 통계만 쓴다. $t$월 수출물가지수와 중량이 붙은 관세청 확정치는 $t+1$월 중순에 나오므로, 1일 시점의 실질화는 $t-1$월 물가로 하고 $t$월 물가·중량 물량은 중순 시점 모형으로 따로 둔다.

In [9]:
# 달력 설명변수 dCAL(§2b)은 그 달의 달력이라 미리 알 수 있으므로 모든 모형에 넣는다.
def bridge(y, x, h, start):
    # y: 목표 Δ12, x: 동월 설명변수 Δ12(없으면 None), h: 알 수 있는 생산의 시차. 표본내 R2와 확장창 표본외 RMSE(%p).
    d = pd.DataFrame({"y": y, "yl": y.shift(h)}).join(dCAL)
    if x is not None: d["x"] = x
    d = d.dropna()
    cols = ["yl"] + CAL_VARS + (["x"] if x is not None else [])
    fit = sm.OLS(d.y, sm.add_constant(d[cols])).fit()
    err = []
    for t in d.index[d.index >= start]:
        tr = d.loc[:t].iloc[:-1]
        m = sm.OLS(tr.y, sm.add_constant(tr[cols])).fit()
        xt = np.r_[1.0, d.loc[t, cols].values.astype(float)]
        err.append(d.loc[t, "y"] - float(m.params.values @ xt))
    e = np.array(err)
    return {"n": len(d), "n_oos": len(e), "R2": fit.rsquared, "coef_x": fit.params.get("x", np.nan), "t_x": fit.tvalues.get("x", np.nan),
            "RMSE": np.sqrt((e ** 2).mean()) * 100}

# t월 수출물가지수는 t+1월 중순에 나온다. t+1월 1일에 쓸 수 있는 실질화는 t-1월 물가로 나눈 것뿐이다.
dRlag = dV - d12(PX).shift(1)
dU = dV - dQ_                                      # 코드 단위 Törnqvist 단가의 Δ12 = 명목 − 물량
dVu = dV - dU.shift(1)                             # 1일 시점: 명목 t − 단가 t-1 (관세청 자료만)
rows = []
for tgt in ["총수출", "비반도체", "반도체"]:
    y = dIP[tgt]
    b0 = bridge(y, None, 1, 201001); b1 = bridge(y, dR[tgt], 1, 201001); b2 = bridge(y, dV[tgt], 1, 201001); b3 = bridge(y, dRlag[tgt], 1, 201001)
    b4 = bridge(y, dQ_[tgt], 1, 201001)     # 중량 Törnqvist 물량 — 물가지수 없이 관세청 자료만으로. 중량은 월간 확정치(익월 중순)에 있다
    b5 = bridge(y, dVu[tgt], 1, 201001)     # 1일 시점, 관세청 자료만: 명목 t − 전월 단가
    for lab, b in [("생산 t-1", b0), ("생산 t-1 + 명목 수출 t", b2), ("생산 t-1 + 명목 수출 t − 단가 t-1", b5), ("생산 t-1 + 물량 수출 t (중량)", b4),
                   ("생산 t-1 + 실질 수출 t (t-1월 물가)", b3), ("생산 t-1 + 실질 수출 t (t월 물가)", b1)]:
        rows.append({"생산지수": tgt, "시점": "t+1월 중순" if ("물량" in lab or "t월 물가" in lab) else "t+1월 1일", "정보": lab, "n": b["n"], "표본외 n": b["n_oos"], "R2": b["R2"], "RMSE": b["RMSE"], "수출 계수": b["coef_x"], "t": b["t_x"]})
TAB_BRIDGE = pd.DataFrame(rows)
# 같은 달 수출 대신 전월 수출(t+1월 1일에 이미 확정치가 있다)을 넣은 모형: 같은 달 수출을 쓰는 설계의 확인
rowsl = []
for tgt in ["총수출", "비반도체", "반도체"]:
    y = dIP[tgt]
    for lab, x in [("생산 t-1 + 명목 수출 t-1", dV[tgt].shift(1)), ("생산 t-1 + 물량 수출 t-1 (중량)", dQ_[tgt].shift(1))]:
        b = bridge(y, x, 1, 201001)
        rowsl.append({"생산지수": tgt, "시점": "t+1월 1일", "정보": lab, "n": b["n"], "표본외 n": b["n_oos"], "R2": b["R2"], "RMSE": b["RMSE"], "수출 계수": b["coef_x"], "t": b["t_x"]})
TAB_BRIDGE = pd.concat([TAB_BRIDGE, pd.DataFrame(rowsl)], ignore_index=True)

# 설명변수를 여러 개 받는 판. 기준 모형에 경기선행지수(t-1)를 더한 둘째 기준(논문 표 6)에 쓴다 — 선행지수는 산업활동동향과 같은 날 나온다.
def bridge2(y, X, h, start):
    # X: 동월 설명변수 DataFrame(없으면 None). 여러 열을 허용한다.
    d = pd.DataFrame({"y": y, "yl": y.shift(h)}).join(dCAL)
    xcols = []
    if X is not None:
        for c in X.columns: d[c] = X[c]; xcols.append(c)
    d = d.dropna(); cols = ["yl"] + CAL_VARS + xcols
    fit = sm.OLS(d.y, sm.add_constant(d[cols])).fit()
    err = []
    for t in d.index[d.index >= start]:
        tr = d.loc[:t].iloc[:-1]; m = sm.OLS(tr.y, sm.add_constant(tr[cols])).fit()
        err.append(d.loc[t, "y"] - float(m.params.values @ np.r_[1.0, d.loc[t, cols].values.astype(float)]))
    e = np.array(err); xc = xcols[-1] if xcols else None
    return {"n": len(d), "n_oos": len(e), "R2": fit.rsquared, "coef_x": fit.params.get(xc, np.nan) if xc else np.nan,
            "t_x": fit.tvalues.get(xc, np.nan) if xc else np.nan, "RMSE": np.sqrt((e ** 2).mean()) * 100}

# 경기선행지수 Δ12 (DT_1C8015 A00)
cli = con.execute("SELECT yyyymm, value FROM fact_cli WHERE tbl='DT_1C8015' AND cli_cd='A00' ORDER BY 1").df().set_index("yyyymm").value
dCLI = np.log(cli).diff(12)

# 월간 모형(2010.01~ 창)에 선행지수 기준을 더한다. 논문 표 6의 둘째 기준. 선행지수 t-1은 t+1월 1일에 이미 나와 있다.
rowsc = []
for tgt in ["총수출", "비반도체", "반도체"]:
    y = dIP[tgt]
    X1 = pd.DataFrame({"cli": dCLI.shift(1).reindex(y.index)})
    b = bridge2(y, X1, 1, 201001)
    rowsc.append({"생산지수": tgt, "시점": "t+1월 1일", "정보": "생산 t-1 + 선행지수 t-1", "n": b["n"], "표본외 n": b["n_oos"], "R2": b["R2"], "RMSE": b["RMSE"], "수출 계수": np.nan, "t": np.nan})
    for lab, x, when in [("명목 수출 t", dV[tgt], "t+1월 1일"), ("명목 수출 t − 단가 t-1", dVu[tgt], "t+1월 1일"), ("물량 수출 t (중량)", dQ_[tgt], "t+1월 중순")]:
        b = bridge2(y, X1.assign(x=x.reindex(y.index)), 1, 201001)
        rowsc.append({"생산지수": tgt, "시점": when, "정보": f"생산 t-1 + 선행지수 t-1 + {lab}", "n": b["n"], "표본외 n": b["n_oos"], "R2": b["R2"], "RMSE": b["RMSE"], "수출 계수": b["coef_x"], "t": b["t_x"]})
TAB_BRIDGE = pd.concat([TAB_BRIDGE, pd.DataFrame(rowsc)], ignore_index=True)
TAB_BRIDGE.to_csv(os.path.join(OUT, "tab2_bridge.csv"), index=False, encoding="utf-8-sig")
print(TAB_BRIDGE.round(3).to_string(index=False))

생산지수      시점                                   정보   n  표본외 n    R2  RMSE  수출 계수      t
 총수출 t+1월 1일                               생산 t-1 366    199 0.795 3.034    NaN    NaN
 총수출 t+1월 1일                     생산 t-1 + 명목 수출 t 366    199 0.812 2.846  0.086  5.690
 총수출 t+1월 1일            생산 t-1 + 명목 수출 t − 단가 t-1 366    199 0.812 2.911  0.136  5.732
 총수출 t+1월 중순                생산 t-1 + 물량 수출 t (중량) 366    199 0.812 2.835  0.151  5.703
 총수출 t+1월 1일           생산 t-1 + 실질 수출 t (t-1월 물가) 366    199 0.826 2.785  0.201  8.021
 총수출 t+1월 중순             생산 t-1 + 실질 수출 t (t월 물가) 366    199 0.812 2.850  0.159  5.761
비반도체 t+1월 1일                               생산 t-1 366    199 0.744 3.232    NaN    NaN
비반도체 t+1월 1일                     생산 t-1 + 명목 수출 t 366    199 0.781 2.800  0.124  7.804
비반도체 t+1월 1일            생산 t-1 + 명목 수출 t − 단가 t-1 366    199 0.764 3.068  0.128  5.462
비반도체 t+1월 중순                생산 t-1 + 물량 수출 t (중량) 366    199 0.767 2.971  0.153  5.881
비반도체 t+1월 1일           생산 t-1 + 실질 수출 t (t-

## §5. 그림 1·2

그림 1은 총수출의 명목 수출, Törnqvist 연쇄 물량, 광공업생산지수를 2020=100으로 맞춰 로그 축에 놓은 것이다(12개월 이동평균).
그림 2는 물량과 생산지수 $\Delta_{12}$의 60개월 이동 상관이다(총수출, 비반도체, 반도체).

In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "figure.dpi": 100})
def style(ax):
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.grid(True, color="0.87", linewidth=0.8); ax.set_axisbelow(True)
def tdate(idx): return pd.to_datetime(pd.Index(idx).astype(int).astype(str), format="%Y%m")
def rebase(s):
    m = s.rolling(12).mean(); return m / m.loc[202001:202012].mean() * 100

def fig_group(g, fname):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(tdate(W.index), rebase(W[g]).values, "--", color="0.55", lw=1.2, label="명목 수출(달러)")
    ax.plot(tdate(QT.index), rebase(QT[g]).values, "-", color="black", lw=1.8, label="수출 물량(Tornqvist 연쇄)")
    ax.plot(tdate(IPX.index), rebase(IPX[g]).values, "-", color="0.55", lw=1.8, label="생산지수")
    ax.set_yscale("log"); ax.set_ylabel("2020=100, 12개월 이동평균 (로그 축)")
    from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
    ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000) if ax.get_ylim()[0] <= t <= ax.get_ylim()[1]]
    ax.yaxis.set_major_locator(FixedLocator(ticks)); ax.yaxis.set_major_formatter(FixedFormatter([str(t) for t in ticks]))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.legend(frameon=False, ncol=4, loc="upper left"); style(ax); fig.tight_layout()
    fig.savefig(os.path.join(IMG, fname), dpi=300); plt.close(fig)
fig_group("총수출", "fig_b1_exports_ip.png")

# 그림 2: 물량 대 생산지수 Δ12의 60개월 이동 상관 — 동조성이 언제 강해지고 약해졌는지는 표로 안 보인다
ROLL = pd.DataFrame({g: eQ[g].rolling(60).corr(eIP[g]) for g in ["총수출", "비반도체", "반도체"]}).dropna(how="all")   # 물량 대 생산, 달력 뺀 잔차
ROLL_R = pd.DataFrame({g: eR[g].rolling(60).corr(eIP[g]) for g in ["총수출", "비반도체", "반도체"]}).dropna(how="all")  # 실질(참고)
ROLL.to_csv(os.path.join(OUT, "tab2_rolling_corr.csv"), encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(10, 4.2))
for g, ls, lw, col in [("총수출", "-", 1.8, "black"), ("비반도체", "--", 1.4, "black"), ("반도체", "-", 1.0, "0.55")]:
    ax.plot(tdate(ROLL.index), ROLL[g].values, ls, lw=lw, color=col, label=g)
for p, a, b in PHASES[1:]:
    ax.axvline(pd.Timestamp(f"{a}-01-01"), color="black", lw=0.8, ls=":")
ax.axhline(0, color="black", lw=0.8); ax.set_ylim(-0.6, 1.0)
ax.set_ylabel("60개월 이동 상관 (수출 물량 대 생산지수, Δ12)"); ax.legend(frameon=False, ncol=3, loc="lower left")
style(ax); fig.tight_layout(); fig.savefig(os.path.join(IMG, "fig_b3_rolling_corr.png"), dpi=300); plt.close(fig)
print("그림 저장:", sorted(f for f in os.listdir(IMG) if f.startswith("fig_b")))
print(ROLL.loc[[200012, 200512, 201012, 201512, 201912, 202312, YM1]].round(2).to_string())
print("이동 상관 최소/최대 (총수출):", ROLL["총수출"].idxmin(), round(ROLL["총수출"].min(), 2), ROLL["총수출"].idxmax(), round(ROLL["총수출"].max(), 2))

그림 저장: ['fig_b1_exports_ip.png', 'fig_b3_rolling_corr.png']
         총수출  비반도체   반도체
yyyymm                  
200012 -0.01 -0.04  0.13
200512  0.77  0.64  0.42
201012  0.81  0.86  0.26
201512  0.69  0.66  0.28
201912  0.22  0.43 -0.22
202312  0.65  0.79  0.43
202607  0.53  0.22  0.39
이동 상관 최소/최대 (총수출): 200012 -0.01 201406 0.89


## §6. 검증

구조 조건은 `assert`로, 본문 인용 수치는 `chk()`로 대조한다. 어긋나면 마지막에 멈춘다. 본문 수치는 집필하면서 채운다.

In [11]:
FAIL = []; N_CHK = [0]
def chk(name, calc, doc, nd=1):
    N_CHK[0] += 1
    c, d = round(float(calc), nd), round(float(doc), nd)
    if abs(c - d) > 10**(-nd) / 2 + 1e-9:
        FAIL.append(f"{name}: 계산 {c} ≠ 본문 {d}")

# 구조
assert R["총수출"].first_valid_index() == 199501 and R["총수출"].notna().all()
assert R["의약품"].first_valid_index() == 201512 and R["제조장비"].first_valid_index() == 200501
assert R["선박"].isna().all()
assert (IPX.loc[:, [g for g in MAP2]].notna().all()).all(), IPX.columns[IPX.isna().any()]
assert (PX["비반도체"] > 0).all() and R["비반도체"].notna().all()
assert all(KW[c] > 0 for a, b, _, _, _ in MAP2.values() for c in a + b)
assert all(XW[c] > 0 for _, _, c, d, _ in MAP2.values() for c in c + d)
# 비반도체 지수가 총지수와 반도체 사이에서 벗어나지 않는 방향으로 움직이는가 (합성의 부호 점검)
assert np.corrcoef(d12(IPX["비반도체"]).dropna(), d12(IPX["총수출"]).loc[d12(IPX["비반도체"]).dropna().index])[0, 1] > 0.8
assert TAB_REAL.shape[0] == len(PHASES) * len(MAP2) and TAB_SYNC2.shape[0] == len(MAP2)
assert TAB_SYNC2.loc["총수출", "물량·생산"] > 0.5, TAB_SYNC2.loc["총수출", "물량·생산"]

# ---- 본문 인용 수치: trend-vs-industrial-production.md (성질별 통계 + 생산지수) ----
PH = list(PH_LABEL.values())
def real(g, ph): return TAB_REAL[(TAB_REAL.계열 == g) & (TAB_REAL.국면 == ph)].iloc[0]
TF = TAB_FULL.copy(); TF["명목−생산"] = TF["명목"] - TF["생산지수"]
# 표 3
T3 = {"총수출": [6.5, 4.5, 2.0, 4.4, -2.5, 2.1], "비반도체": [5.6, 3.0, 2.5, 1.9, 0.6, 3.6], "반도체": [9.5, 9.2, 0.3, 19.8, -19.5, -10.3], "화공품": [6.9, 1.4, 5.5, 1.9, 3.6, 5.0], "승용차": [7.7, 1.7, 5.9, 2.3, 3.7, 5.4], "일반기계": [6.3, 1.5, 4.8, 2.8, 2.0, 3.5], "석유제품": [10.2, 5.1, 5.1, 2.1, 3.1, 8.1], "철강제품": [5.1, 1.7, 3.5, 1.5, 2.0, 3.6], "선박": [5.9, 7.3, -1.4, 4.8, -6.2, 1.1], "자동차부품": [10.0, 1.0, 9.0, 5.1, 3.9, 4.9], "무선통신기기": [9.7, 8.5, 1.2, 3.8, -2.7, 5.8], "컴퓨터주변기기": [6.7, 10.9, -4.2, 2.9, -7.0, 3.8], "정밀기기": [8.7, 2.5, 6.2, 2.9, 3.2, 5.8], "이차전지": [9.3, 2.4, 6.9, 7.5, -0.5, 1.9], "제조장비": [27.7, -5.9, 33.6, 2.8, 30.7, 24.9], "의약품": [12.1, 8.1, 4.0, 6.3, -2.3, 5.8], "가전제품": [-0.5, 2.3, -2.9, -0.8, -2.1, 0.2]}
for g, vals in T3.items():
    for col, v in zip(["명목", "단가", "물량", "생산지수", "물량−생산", "명목−생산"], vals): chk(f"표3 {g} {col}", TF.loc[g, col], v)
chk("개월 수", len(W), 379, 0); chk("소분류 수", con.execute("SELECT COUNT(*) FROM dim_ksic WHERE level=3").fetchone()[0], 81, 0)
chk("소분류 1995부터", con.execute("SELECT COUNT(*) FROM (SELECT f.ksic, MIN(f.yyyymm) m0 FROM fact_ip f JOIN dim_ksic d USING (ksic) WHERE d.level=3 AND f.measure='prod' GROUP BY 1) WHERE m0=199501").fetchone()[0], 76, 0)
# III.1: 반도체 중량이 3분의 1, 그 대부분이 금액 1%인 기타 개별소자(44307)
chk("III.1 반도체 중량 2024/2023", Q.loc[202401:202412, "반도체"].sum() / Q.loc[202301:202312, "반도체"].sum(), 0.33, 2)
_w = raw[raw.temper_cd == "44307"].groupby(raw.yyyymm // 100)[["dlr", "wgt"]].sum()
_m = raw[raw.temper_cd == "44301"].groupby(raw.yyyymm // 100)[["dlr", "wgt"]].sum()
_Q23, _W23 = Q.loc[202301:202312, "반도체"].sum(), W.loc[202301:202312, "반도체"].sum()
chk("III.1 44307 중량 비중 2023 %", _w.loc[2023, "wgt"] / _Q23 * 100, 87, 0); chk("III.1 44307 금액 비중 2023 %", _w.loc[2023, "dlr"] / _W23 * 100, 2.5)
chk("III.1 메모리 금액 비중 2023 %", _m.loc[2023, "dlr"] / _W23 * 100, 51, 0); chk("III.1 메모리 중량 비중 2023 %", _m.loc[2023, "wgt"] / _Q23 * 100, 2.6)
chk("III.1 44307 중량 2023 만톤", _w.loc[2023, "wgt"] / 1e7, 18.6); chk("III.1 44307 중량 2024 만톤", _w.loc[2024, "wgt"] / 1e7, 4.1)
chk("III.1 메모리 중량 2023 백톤", _m.loc[2023, "wgt"] / 1e5, 56, 0); chk("III.1 메모리 중량 2024 백톤", _m.loc[2024, "wgt"] / 1e5, 55, 0)
# III.1: 총수출 물량 31년 연평균, 끝점 비교 대 월 연쇄
chk("III.1 총수출 물량 끝점", TF.loc["총수출", "물량"], 2.0)
chk("III.1 총수출 물량 월 연쇄", np.log(QT["총수출"].loc[202508:202607].mean() / QT["총수출"].loc[199501:199512].mean()) * 100 / yrs(1995, 2026), 3.0)
# 그림 1 본문 (연평균 수준의 배율)
Qy2 = annual(QT, "mean")
def ratio(T, g, y1, y0): return T.loc[y1, g] / T.loc[y0, g]
chk("생산 2008/1995", ratio(IPy, "총수출", 2008, 1995), 2.5); chk("물량 2008/1995", ratio(Qy2, "총수출", 2008, 1995), 2.1)
chk("생산 2026/2008", ratio(IPy, "총수출", 2026, 2008), 1.6); chk("물량 2026/2008", ratio(Qy2, "총수출", 2026, 2008), 1.2)
chk("생산 2026/2016", ratio(IPy, "총수출", 2026, 2016), 1.21, 2); chk("물량 2026/2016", ratio(Qy2, "총수출", 2026, 2016), 1.12, 2)
chk("명목 2008/1995", ratio(Vy, "총수출", 2008, 1995), 3.4)
# IV.2: 총수출의 Δ12 변동 가운데 달력 설명변수가 설명하는 몫 (%)
chk("IV.2 달력 몫 총수출 물량", CAL_R2.loc["총수출", "물량"] * 100, 19, 0); chk("IV.2 달력 몫 총수출 명목", CAL_R2.loc["총수출", "명목"] * 100, 4, 0)
# 표 4 (달력 효과를 뺀 Δ12 상관)
T4 = {"총수출": [0.54, 0.63, -0.01, 0.72, 0.80, 0.05, 0.71, 0.36], "비반도체": [0.46, 0.64, -0.04, 0.60, 0.83, 0.46, 0.81, 0.38], "반도체": [0.35, 0.51, 0.13, 0.48, 0.32, -0.38, 0.55, 0.13],
      "화공품": [0.22, 0.56, -0.46, 0.24, 0.48, 0.51, 0.64, 0.45], "승용차": [0.66, 0.72, 0.48, 0.75, 0.87, 0.87, 0.81, 0.79], "일반기계": [0.45, 0.55, 0.01, 0.54, 0.86, 0.38, 0.45, 0.19],
      "석유제품": [0.56, 0.40, 0.56, 0.38, 0.60, 0.28, 0.67, 0.86], "철강제품": [0.20, 0.41, -0.57, 0.25, 0.62, 0.42, 0.52, 0.28], "선박": [0.18, 0.18, 0.27, -0.03, 0.25, 0.23, 0.29, 0.12],
      "자동차부품": [0.54, 0.56, 0.46, 0.13, 0.87, 0.47, 0.77, 0.32], "무선통신기기": [0.45, 0.48, 0.40, 0.69, 0.41, -0.20, 0.45, -0.19], "컴퓨터주변기기": [0.45, 0.27, 0.23, -0.09, 0.70, 0.64, 0.51, -0.39],
      "정밀기기": [0.11, 0.18, 0.23, -0.17, 0.37, 0.50, 0.26, -0.11], "이차전지": [0.17, 0.51, 0.10, -0.07, 0.59, 0.18, -0.16, -0.51], "제조장비": [0.16, 0.17, -0.13, 0.56, 0.59, 0.51, 0.16, 0.19],
      "의약품": [0.04, 0.13, 0.19, -0.26, 0.07, 0.34, -0.13, 0.25], "가전제품": [0.41, 0.46, 0.71, 0.23, 0.52, -0.18, 0.51, 0.71]}
for g, vals in T4.items():
    for col, v in zip(["물량·생산", "명목·생산"] + PH, vals): chk(f"표4 {g} {col}", TAB_SYNC2.loc[g, col], v, 2)
chk("표4 n", TAB_SYNC2.loc["총수출", "n"], 367, 0)
# IV.2 본문: 물량과 명목의 차이가 0.1을 넘는 계열
_dd = TAB_SYNC2["물량·생산"] - TAB_SYNC2["명목·생산"]
if sorted(_dd[_dd > 0.1].index) != sorted(["석유제품", "컴퓨터주변기기"]): FAIL.append(f"IV.2 물량이 0.1 넘게 높은 계열 {list(_dd[_dd > 0.1].index)}")
if sorted(_dd[_dd < -0.1].index) != sorted(["비반도체", "반도체", "화공품", "철강제품", "이차전지", "일반기계"]): FAIL.append(f"IV.2 명목이 0.1 넘게 높은 계열 {list(_dd[_dd < -0.1].index)}")
if TAB_SYNC2.loc[EXP15, "물량·생산"].idxmax() != "승용차" or TAB_SYNC2.loc[EXP15, "명목·생산"].idxmax() != "승용차": FAIL.append("IV.2 승용차가 가장 높음")
if sorted(TAB_SYNC2.loc[EXP15, "물량·생산"].nsmallest(5).index) != sorted(["의약품", "정밀기기", "제조장비", "이차전지", "선박"]): FAIL.append("IV.2 물량 기준 낮은 다섯")
if not all(TAB_SYNC2.loc[g, p] > 0.7 for g in ["총수출"] for p in (PH[1], PH[2], PH[4])): FAIL.append("IV.2 총수출 0.7 넘는 국면")
if TAB_SYNC2.loc["반도체", PH].max() > 0.55 + 0.005: FAIL.append("IV.2 반도체 0.55를 넘지 않음")
# 그림 2 본문 (달력 효과를 뺀 이동 상관)
chk("이동상관 총수출 첫 값", ROLL["총수출"].iloc[0], -0.01, 2); chk("이동상관 총수출 첫 시점", ROLL["총수출"].index[0], 200012, 0)
chk("이동상관 총수출 0.5 넘는 첫 달", ROLL.index[ROLL["총수출"] > 0.5][0], 200305, 0)
chk("이동상관 총수출 최대", ROLL["총수출"].max(), 0.89, 2); chk("이동상관 총수출 최대 시점", ROLL["총수출"].idxmax(), 201406, 0)
chk("이동상관 총수출 2016~19 최저", ROLL.loc[201601:201912, "총수출"].min(), 0.19, 2); chk("이동상관 총수출 2016~19 최저 시점", ROLL.loc[201601:201912, "총수출"].idxmin(), 201909, 0)
chk("이동상관 총수출 끝", ROLL["총수출"].iloc[-1], 0.53, 2)
chk("이동상관 비반도체 최소", ROLL["비반도체"].min(), -0.05, 2); chk("이동상관 비반도체 최소 시점", ROLL["비반도체"].idxmin(), 200210, 0)
chk("이동상관 비반도체 0.5 넘는 첫 달", ROLL.index[ROLL["비반도체"] > 0.5][0], 200410, 0)
chk("이동상관 비반도체 최대", ROLL["비반도체"].max(), 0.89, 2); chk("이동상관 비반도체 최대 시점", ROLL["비반도체"].idxmax(), 201406, 0); chk("이동상관 비반도체 끝", ROLL["비반도체"].iloc[-1], 0.22, 2)
chk("이동상관 반도체 최대", ROLL["반도체"].max(), 0.76, 2); chk("이동상관 반도체 최대 시점", ROLL["반도체"].idxmax(), 200112, 0)
chk("이동상관 반도체 최소", ROLL["반도체"].min(), -0.35, 2); chk("이동상관 반도체 최소 시점", ROLL["반도체"].idxmin(), 202008, 0); chk("이동상관 반도체 끝", ROLL["반도체"].iloc[-1], 0.39, 2)
# IV.2 선후행: 달력 효과를 뺀 Δ12 교차상관
_lq, _lv = TAB_XC["물량 최대 시차"], TAB_XC["명목 최대 시차"]
chk("IV.2 물량 최대 시차 0인 계열 수", (_lq == 0).sum(), 13, 0); chk("IV.2 계열 수", len(TAB_XC), 17, 0)
for g, k in [("화공품", -9), ("철강제품", -6), ("제조장비", -4), ("의약품", 5)]: chk(f"IV.2 물량 최대 시차 {g}", _lq[g], k, 0)
_ex = TAB_XC.loc[_lq[_lq != 0].index]
chk("IV.2 예외 동시 상관 최소", _ex["물량 동시 상관"].min(), 0.04, 2); chk("IV.2 예외 동시 상관 최대", _ex["물량 동시 상관"].max(), 0.22, 2)
chk("IV.2 예외 최대 상관 최소", _ex["물량 최대 상관"].min(), 0.12, 2); chk("IV.2 예외 최대 상관 최대", _ex["물량 최대 상관"].max(), 0.32, 2)
chk("IV.2 명목 최대 시차 음수 계열 수", (_lv < 0).sum(), 7, 0); chk("IV.2 명목 최대 시차 0 계열 수", (_lv == 0).sum(), 10, 0)
if _lv["총수출"] >= 0: FAIL.append("IV.2 총수출 명목 최대 시차 음수")
if (_lv > 0).any() or list(_lq[_lq > 0].index) != ["의약품"]: FAIL.append("IV.2 양의 시차 계열")
# 표 5 (브리지 회귀, 모든 모형에 달력)
def br(tab, tgt, info):
    r = tab[(tab.생산지수 == tgt) & (tab.정보 == info)]; assert len(r) == 1, (tgt, info); return r.iloc[0]
def red(tab, g, base, info): return (1 - br(tab, g, info).RMSE / br(tab, g, base).RMSE) * 100
B1, B2 = "생산 t-1", "생산 t-1 + 선행지수 t-1"
_N, _P, _Q = " + 명목 수출 t", " + 명목 수출 t − 단가 t-1", " + 물량 수출 t (중량)"
T5 = {"총수출": [(0.79, 3.03), (0.81, 0.09, 5.7, 2.85, -6), (0.81, 0.14, 5.7, 2.91, -4), (0.81, 0.15, 5.7, 2.84, -7),
                 (0.83, 2.91), (0.84, 0.09, 6.4, 2.87, -1), (0.85, 0.15, 7.0, 2.73, -6), (0.85, 0.19, 7.9, 2.59, -11)],
      "비반도체": [(0.74, 3.23), (0.78, 0.12, 7.8, 2.80, -13), (0.76, 0.13, 5.5, 3.07, -5), (0.77, 0.15, 5.9, 2.97, -8),
                  (0.78, 3.01), (0.82, 0.12, 8.5, 2.60, -14), (0.80, 0.12, 5.6, 2.83, -6), (0.81, 0.16, 6.7, 2.70, -10)],
      "반도체": [(0.79, 9.41), (0.80, 0.07, 3.7, 9.83, 4), (0.81, 0.14, 6.1, 9.20, -2), (0.80, 0.10, 3.9, 9.39, 0),
                (0.79, 9.38), (0.80, 0.06, 3.1, 9.81, 5), (0.81, 0.14, 5.9, 9.21, -2), (0.80, 0.10, 3.9, 9.37, 0)]}
for g, rows_ in T5.items():
    for k, base in enumerate((B1, B2)):
        for m_, suf in enumerate(["", _N, _P, _Q]):
            v = rows_[k * 4 + m_]; info = base + suf; r = br(TAB_BRIDGE, g, info)
            chk(f"표5 {g} {info} R2", r.R2, v[0], 2)
            if suf == "": chk(f"표5 {g} {info} RMSE", r.RMSE, v[1], 2); continue
            chk(f"표5 {g} {info} 계수", r["수출 계수"], v[1], 2); chk(f"표5 {g} {info} t", r.t, v[2], 1)
            chk(f"표5 {g} {info} RMSE", r.RMSE, v[3], 2); chk(f"표5 {g} {info} 감소%", -red(TAB_BRIDGE, g, base, info), v[4], 0)
chk("표5 n", br(TAB_BRIDGE, "총수출", B1).n, 366, 0); chk("표5 표본외 n", br(TAB_BRIDGE, "총수출", B1)["표본외 n"], 199, 0)
# IV.3 본문
chk("IV.3 선행지수 기준이 첫째 기준보다 낮춘 %", red(TAB_BRIDGE, "총수출", B1, B2), 4, 0)
chk("IV.3 총수출 전월 명목 RMSE", br(TAB_BRIDGE, "총수출", "생산 t-1 + 명목 수출 t-1").RMSE, 3.11, 2)
chk("IV.3 총수출 전월 물량 RMSE", br(TAB_BRIDGE, "총수출", "생산 t-1 + 물량 수출 t-1 (중량)").RMSE, 3.02, 2)
if not all(red(TAB_BRIDGE, g, B1, i) < 1 for g in ["총수출", "비반도체", "반도체"] for i in ["생산 t-1 + 명목 수출 t-1", "생산 t-1 + 물량 수출 t-1 (중량)"]): FAIL.append("IV.3 전월 수출이 오차를 1% 넘게 줄임")
_semi_t = [br(TAB_BRIDGE, "반도체", b + i).t for b in (B1, B2) for i in (_N, _P, _Q)]
chk("IV.3 반도체 수출 계수 t 최소", min(_semi_t), 3.1, 1); chk("IV.3 반도체 수출 계수 t 최대", max(_semi_t), 6.1, 1)
if max(red(TAB_BRIDGE, "반도체", b, b + i) for b in (B1, B2) for i in (_N, _P, _Q)) >= 3: FAIL.append("V 반도체 3% 넘게 줄지 않음")

# ---- 2026-09-14 추가: 본문 산문 수치와 서술의 조건 ----
# IV.1 (2026-09-30): 물량지수가 생산보다 느린/빠른 품목군 수, 가장 느린 셋, 단가 9% 넘는 둘
_gq = TF.loc[EXP15, "물량−생산"]
chk("IV.1 물량이 생산보다 느린 품목군 수", (_gq < 0).sum(), 7, 0); chk("IV.1 빠른 품목군 수", (_gq > 0).sum(), 8, 0)
if list(_gq.nsmallest(3).index) != ["반도체", "컴퓨터주변기기", "선박"]: FAIL.append(f"IV.1 가장 느린 셋 {list(_gq.nsmallest(3).index)}")
if sorted(TF.loc[["반도체", "컴퓨터주변기기", "선박"], "단가"].pipe(lambda x: x[x > 9]).index) != ["반도체", "컴퓨터주변기기"]: FAIL.append("IV.1 단가 9% 넘는 둘")
# II.2: 비반도체 생산지수에서 빠진 코드
_drop_raw = [c for c in EX_ALL if c not in EX_SEMI_CODES]
if _drop_raw != ["C34"]: FAIL.append(f"II.2 원지수 뺀 코드 {_drop_raw}")
# IV.2 표 4: 국면별 첫 열은 Δ12가 1996.01부터
chk("IV.2 표4 Δ12 첫 달", dQ_["총수출"].dropna().index[0], 199601, 0)

print(f"검증 {N_CHK[0]}건, 어긋남 {len(FAIL)}건")
for f in FAIL: print("  ", f)
assert not FAIL

검증 405건, 어긋남 0건
